In [2]:
from ultralytics import YOLO
import torch
import time
import os
import pandas as pd

# =====================================================
# DEVICE
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# =====================================================
# MODEL PATHS
# Replace these with your ACTUAL paths
# =====================================================

model_paths = {
    "YOLOv8": [
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Clear\YOLOv8(NO CBAM)\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Low\YOLOv8(NO CBAM)\train24\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Medium\YOLOv8(NO CBAM)\train32\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\High\YOLOv8(NO CBAM)\train41\weights\best.pt"
    ],

    "YOLO11": [
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Clear\YOLO11(NO CBAM)\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Low\YOLO11(NO CBAM)\train31\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Medium\YOLO11(NO CBAM)\train39\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\High\YOLO11(NO CBAM)\train48\weights\best.pt"
    ],

    "Early Backbone": [
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Clear\yolov8-CBAM-earlyBBone\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Low\yolov8-CBAM-earlyBBone\train27\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Medium\yolov8-CBAM-earlyBBone\train35\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\High\yolov8-CBAM-earlyBBone\train44\weights\best.pt"
    ],

    "Late Backbone": [
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Clear\yolov8-CBAM-lateBBone\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Low\yolov8-CBAM-lateBBone\train26\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Medium\yolov8-CBAM-lateBBone\train34\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\High\yolov8-CBAM-lateBBone\train43\weights\best.pt"
    ],

    "Early + Late Backbone": [
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Clear\yolov8-CBAM-early&lateBBone\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Low\yolov8-CBAM-early&lateBBone\train25\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Medium\yolov8-CBAM-early&lateBBone\train33\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\High\yolov8-CBAM-early&lateBBone\train42\weights\best.pt"
    ],

    "Neck Only": [
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Clear\yolov8-CBAM-neckOnly\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Low\yolov8-CBAM-neckOnly\train28\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Medium\yolov8-CBAM-neckOnly\train36\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\High\yolov8-CBAM-neckOnly\train45\weights\best.pt"
    ],

    "Pre-Detect": [
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Clear\yolov8-CBAM-preDetect\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Low\yolov8-CBAM-preDetect\train29\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Medium\yolov8-CBAM-preDetect\train37\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\High\yolov8-CBAM-preDetect\train46\weights\best.pt"
    ],

    "Neck + Backbone": [
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Clear\yolov8-CBAM-neck&BBone\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Low\yolov8-CBAM-neck&BBone\train30\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\Medium\yolov8-CBAM-neck&BBone\train38\weights\best.pt",
        r"C:\Clone Repos\Modified-Ultralytics-for-CBAM-Implementation-of-YOLOv8\tests\runs\detect\SeparateTurbidityDatasets\High\yolov8-CBAM-neck&BBone\train47\weights\best.pt"
    ]
}

# =====================================================
# BENCHMARK SETTINGS
# =====================================================

iterations = 100
warmup_runs = 50

input_tensor = torch.randn(1, 3, 640, 640).to(device)

results = []

# =====================================================
# BENCHMARK LOOP
# =====================================================

for model_name, paths in model_paths.items():

    fps_list = []
    latency_list = []

    print(f"\nBenchmarking: {model_name}")

    for path in paths:

        # -------------------------
        # Load model
        # -------------------------
        yolo_model = YOLO(path)
        model = yolo_model.model

        model.to(device)
        model.eval()

        # -------------------------
        # Warmup
        # -------------------------
        with torch.no_grad():
            for _ in range(warmup_runs):
                _ = model(input_tensor)

        # -------------------------
        # Timing
        # -------------------------
        total_time = 0

        with torch.no_grad():

            for _ in range(iterations):

                if device.type == "cuda":
                    torch.cuda.synchronize()

                start = time.time()

                _ = model(input_tensor)

                if device.type == "cuda":
                    torch.cuda.synchronize()

                end = time.time()

                total_time += (end - start)

        # -------------------------
        # Metrics
        # -------------------------
        avg_latency = (total_time / iterations) * 1000
        fps = 1000 / avg_latency

        fps_list.append(fps)
        latency_list.append(avg_latency)

        print(f"{os.path.basename(path)}")
        print(f"Latency: {avg_latency:.2f} ms")
        print(f"FPS: {fps:.2f}")

    # =================================================
    # AVERAGE ACROSS TURBIDITIES
    # =================================================

    avg_model_fps = sum(fps_list) / len(fps_list)
    avg_model_latency = sum(latency_list) / len(latency_list)

    results.append({
        "Model": model_name,
        "Average Latency (ms)": round(avg_model_latency, 2),
        "Average FPS": round(avg_model_fps, 2)
    })

# =====================================================
# FINAL TABLE
# =====================================================

df = pd.DataFrame(results)

print("\n==============================")
print("AVERAGED RESULTS")
print("==============================")

print(df)

Using device: cuda

Benchmarking: YOLOv8
best.pt
Latency: 8.24 ms
FPS: 121.33
best.pt
Latency: 9.39 ms
FPS: 106.50
best.pt
Latency: 8.89 ms
FPS: 112.51
best.pt
Latency: 8.63 ms
FPS: 115.90

Benchmarking: YOLO11
best.pt
Latency: 11.42 ms
FPS: 87.53
best.pt
Latency: 11.72 ms
FPS: 85.31
best.pt
Latency: 11.81 ms
FPS: 84.67
best.pt
Latency: 11.42 ms
FPS: 87.54

Benchmarking: Early Backbone
best.pt
Latency: 13.43 ms
FPS: 74.49
best.pt
Latency: 13.34 ms
FPS: 74.96
best.pt
Latency: 13.75 ms
FPS: 72.71
best.pt
Latency: 13.24 ms
FPS: 75.53

Benchmarking: Late Backbone
best.pt
Latency: 13.58 ms
FPS: 73.65
best.pt
Latency: 13.36 ms
FPS: 74.84
best.pt
Latency: 13.45 ms
FPS: 74.33
best.pt
Latency: 13.56 ms
FPS: 73.72

Benchmarking: Early + Late Backbone
best.pt
Latency: 9.72 ms
FPS: 102.85
best.pt
Latency: 9.64 ms
FPS: 103.77
best.pt
Latency: 9.71 ms
FPS: 102.98
best.pt
Latency: 9.90 ms
FPS: 100.97

Benchmarking: Neck Only
best.pt
Latency: 13.08 ms
FPS: 76.43
best.pt
Latency: 13.09 ms
FPS: 76.39
be